# Chapter 21 — Sampling Is Part of the Program

**Book alignment:** Debugging AI From First Principles, Chapter 21

**Question this notebook isolates:** The refund fixture flickers — 9/12 Monday, 5/12
Wednesday, same bundle. Treating the *distribution over outputs* as the object, does an
N≥20 trial series with a fixed signature separate **H1** (sampling spread — a deterministic
probe collapses it to one mode) from **H2** (bimodal competence — two modes persist even at
temperature 0)? And is `pass@k` the shipped reliability?

In [ ]:
import numpy as np

def opaque_gen(*, temperature, case_seed, bimodal=False):
    """Deterministic given (temperature, case_seed). Returns a citation label."""
    rng = np.random.default_rng(case_seed)
    if bimodal:                                   # H2: the system genuinely holds both answers
        return "4.2-exception" if rng.random() < 0.6 else "general-policy"
    # H1: one correct attractor + a temperature-scaled tail of wrong citations
    if rng.random() < 1.0 - 0.45 * temperature:
        return "4.2-exception"
    return rng.choice(["general-policy", "4.1", "truncated"])

def classify(label):
    return "correct" if label == "4.2-exception" else "wrong"

## 1. Fix the classifier, then sweep the baseline (N=20, seeds paired)

In [ ]:
SEEDS = list(range(20))
baseline = [classify(opaque_gen(temperature=0.7, case_seed=s)) for s in SEEDS]
n_correct = baseline.count("correct")
print(f"baseline pass@1: {n_correct}/20   (raw counts before any percentage)")
modes = {m: baseline.count(m) for m in set(baseline)}
print("modes:", modes)
assert n_correct < 20 and n_correct > 5           # flickers - neither always-pass nor always-fail

## 2. Deterministic probe (temp 0) vs a bimodal-world control

In [ ]:
det   = [classify(opaque_gen(temperature=0.0, case_seed=s)) for s in SEEDS[:10]]
det_bimodal = [classify(opaque_gen(temperature=0.0, case_seed=s, bimodal=True)) for s in SEEDS[:10]]
print("H1 world, temp 0 x10:", det, "->", det.count("correct"), "/10")
print("H2 world, temp 0 x10:", det_bimodal, "-> modes", {m: det_bimodal.count(m) for m in set(det_bimodal)})

assert det.count("correct") >= 9                  # H1: collapses to one correct mode
assert min(det_bimodal.count("correct"), det_bimodal.count("wrong")) >= 3   # H2: two modes persist
print("\nour fixture: deterministic probe collapses -> H1 (spread), not H2 (bimodal competence)")

## 3. pass@k is a selection ceiling, not shipped reliability

In [ ]:
def pass_at_k(results, k, c=None):
    n = len(results); c = results.count("correct") if c is None else c
    if n - c < k:
        return 1.0
    return 1.0 - np.prod([(n - c - i) / (n - i) for i in range(k)])   # unbiased estimator

p1 = n_correct / 20
pk = pass_at_k(baseline, k=10)
print(f"pass@1 (shipped)          : {n_correct}/20 = {p1:.2f}")
print(f"pass@10 (selection ceiling): {pk:.2f}   <- what a reranker COULD reach, NOT what ships")
assert pk > p1
print("quoting pass@k as reliability is a category error, not a rounding error")

## What we earned

The debugging object is the distribution, not the draw. A fixed output classifier plus an
N=20 paired-seed series made the shape visible (bimodal at temperature 0.7); the
deterministic probe collapsed it to one correct mode, convicting **H1 (sampling spread)** —
in an H2 world the same probe would leave two modes standing, and no sampling fix would
help. `pass@10` (selection ceiling) sat well above the shipped `pass@1`; reporting the
first as reliability is a category error. A single passing retry is never a verdict.

**Notebook 22 / Chapter 22** takes the surviving failures and asks whether any interior
signal *locates* the veer — strictly as a triage lead, never a verdict.